In [1]:
# Cell 1 — Config, load, and validate the finalized Processed 60-s cohort.
from __future__ import annotations

import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def locate_phase1(start: Path) -> Path:
    """Locate phase1 using the canonical segmented dataset."""
    marker = Path("segmentated_data/dhdata/segments_index.csv")
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / marker).is_file():
            return candidate
        nested = candidate / "phase1"
        if (nested / marker).is_file():
            return nested
    raise FileNotFoundError("Could not locate phase1.")


def require(condition: bool, message: str) -> None:
    """Fail loudly when an integrity condition is not met."""
    if not condition:
        raise AssertionError(message)


PHASE1_DIR = locate_phase1(Path.cwd())
SRC_DIR = PHASE1_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from dataloader.loader import get_data  # noqa: E402

SOURCE_DIR = PHASE1_DIR / "segmentated_data" / "dhdata"
OUTPUT_DIR = PHASE1_DIR / "segmentated_data" / "30s_dhdata"
PARENT_WINDOW_S = 60
CHILD_WINDOW_S = 30
EXPECTED_SESSIONS = 20
STATES = {"Awake": 0, "Drowsy": 1}
DURATION_TOLERANCE_S = 0.25
BOUNDARY_TOLERANCE_S = 1e-9

session_archives = sorted(
    SOURCE_DIR.glob("sample_*.npz"),
    key=lambda path: int(path.stem.split("_")[-1]),
)
require(
    len(session_archives) == EXPECTED_SESSIONS,
    f"Expected 20 source sessions; found {len(session_archives)}.",
)
parent_rows: list[dict[str, object]] = []
parent_signals: dict[str, np.ndarray] = {}
parent_times: dict[str, np.ndarray] = {}
for source_path in session_archives:
    session = source_path.stem
    awake, drowsy = get_data(
        session,
        data_dir=SOURCE_DIR,
        window_sizes=PARENT_WINDOW_S,
        stationarity="processed",
    )
    state_batches = {"Awake": awake[60], "Drowsy": drowsy[60]}
    for state, label in STATES.items():
        batch = state_batches[state]
        signals = np.asarray(batch["processed"], dtype=float)
        times = np.asarray(batch["time"], dtype=float)
        window_ids = np.asarray(batch["window_id"], dtype=int)
        fs = float(batch["fs"])
        require(
            signals.ndim == 2 and signals.shape == times.shape,
            f"Signal/time mismatch: {session}, {state}.",
        )
        require(
            len(signals) == len(window_ids)
            and len(np.unique(window_ids)) == len(window_ids),
            f"Invalid parent IDs: {session}, {state}.",
        )
        require(np.isfinite(fs) and fs > 0, f"Invalid fs: {session}.")
        require(
            abs(signals.shape[1] / fs - 60.0)
            <= DURATION_TOLERANCE_S,
            f"Parent duration is not approximately 60 s: {session}.",
        )
        for index, parent_window_id in enumerate(window_ids):
            signal = signals[index].copy()
            time = times[index].copy()
            parent_uid = (
                f"{session}_{state.lower()}_w{parent_window_id:04d}_60s"
            )
            require(parent_uid not in parent_signals, "Duplicate parent UID.")
            require(
                signal.size > 0
                and np.isfinite(signal).all()
                and np.isfinite(time).all()
                and np.all(np.diff(time) > 0),
                f"Invalid parent signal/time: {parent_uid}.",
            )
            require(
                np.allclose(
                    np.diff(time), 1.0 / fs, rtol=1e-8, atol=1e-10
                ),
                f"Parent time/fs mismatch: {parent_uid}.",
            )
            parent_signals[parent_uid] = signal
            parent_times[parent_uid] = time
            parent_rows.append(
                {
                    "session": session,
                    "session_id": int(session.split("_")[-1]),
                    "state": state,
                    "label": label,
                    "parent_window_id": int(parent_window_id),
                    "parent_window_60_id": parent_uid,
                    "fs": fs,
                    "n_samples_parent": signal.size,
                    "parent_start_time_s": float(time[0]),
                    "parent_end_time_s": float(time[-1]),
                    "source_file": source_path.name,
                }
            )
parent_index = (
    pd.DataFrame(parent_rows)
    .sort_values(["session_id", "label", "parent_window_id"])
    .reset_index(drop=True)
)
require(parent_index["parent_window_60_id"].is_unique, "Duplicate parents.")
require(
    parent_index["session"].nunique() == EXPECTED_SESSIONS,
    "The parent cohort does not contain 20 sessions.",
)
input_state_counts = parent_index["state"].value_counts().reindex(STATES)
fs_counts = (
    parent_index.assign(fs_display=parent_index["fs"].round(6))
    .groupby("fs_display")
    .size()
    .rename("n_parent_windows")
    .reset_index()
)
print("Input: get_data(..., window_sizes=60, stationarity='processed')")
print("No raw reload and no preprocessing")
print(f"Sessions: {parent_index['session'].nunique()}")
print(f"60-s parent windows: {len(parent_index)}")
print(input_state_counts.to_string())
display(fs_counts)


Input: get_data(..., window_sizes=60, stationarity='processed')
No raw reload and no preprocessing
Sessions: 20
60-s parent windows: 901
state
Awake     596
Drowsy    305


,fs_display,n_parent_windows
0,24.989380,55
1,24.991878,46
2,24.992877,42
3,24.993127,48
4,24.993627,48
5,24.993752,91
6,24.993767,38
7,24.995626,34
8,24.995938,42
9,24.996219,44


In [2]:
# Cell 2 — Split and save one packed, processed-only NPZ per session.
def pack_windows(windows: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Pack variable-length numeric windows without padding."""
    lengths = np.asarray([len(window) for window in windows], dtype=int)
    require(np.all(lengths > 0), "Cannot pack an empty child window.")
    offsets = np.concatenate([[0], np.cumsum(lengths)])
    values = np.concatenate(windows)
    return values, offsets


expected_output = PHASE1_DIR / "segmentated_data" / "30s_dhdata"
require(OUTPUT_DIR.resolve() == expected_output.resolve(), "Unsafe output path.")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)
index_rows: list[dict[str, object]] = []
for source_path in session_archives:
    session = source_path.stem
    session_parents = parent_index.loc[
        parent_index["session"].eq(session)
    ].sort_values(["label", "parent_window_id"])
    processed_windows: list[np.ndarray] = []
    time_windows: list[np.ndarray] = []
    child_metadata: list[dict[str, object]] = []
    for parent in session_parents.itertuples(index=False):
        signal = parent_signals[parent.parent_window_60_id]
        time = parent_times[parent.parent_window_60_id]
        relative_time = time - time[0]
        split_index = int(
            np.searchsorted(
                relative_time,
                CHILD_WINDOW_S - BOUNDARY_TOLERANCE_S,
                side="left",
            )
        )
        expected_n30 = int(round(CHILD_WINDOW_S * parent.fs))
        require(
            abs(split_index - expected_n30) <= 1
            and 0 < split_index < len(signal),
            f"Invalid split boundary: {parent.parent_window_60_id}.",
        )
        specs = (
            ("h1", 0.0, 30.0, slice(0, split_index)),
            ("h2", 30.0, 60.0, slice(split_index, len(signal))),
        )
        for subwindow, start_offset, end_offset, child_slice in specs:
            child_signal = signal[child_slice].copy()
            child_time = time[child_slice].copy()
            child_uid = f"{parent.parent_window_60_id}_{subwindow}"
            row_index = len(processed_windows)
            child_window_id = row_index + 1
            processed_windows.append(child_signal)
            time_windows.append(child_time)
            child_metadata.append(
                {
                    "row_index": row_index,
                    "window_id": child_window_id,
                    "window_30_id": child_uid,
                    "parent_window_60_id": parent.parent_window_60_id,
                    "parent_window_id": parent.parent_window_id,
                    "subwindow": subwindow,
                    "start_offset_s": start_offset,
                    "end_offset_s": end_offset,
                    "label": parent.label,
                    "state": parent.state,
                    "start_time": float(child_time[0]),
                    "end_time": float(child_time[-1]),
                    "n_samples": child_signal.size,
                    "source_file": parent.source_file,
                }
            )
    processed_values, processed_offsets = pack_windows(processed_windows)
    time_values, time_offsets = pack_windows(time_windows)
    session_meta = pd.DataFrame(child_metadata)
    require(
        np.array_equal(processed_offsets, time_offsets),
        f"Signal/time packing mismatch: {session}.",
    )
    archive_path = OUTPUT_DIR / f"{session}.npz"
    prefix = str(CHILD_WINDOW_S)
    payload = {
        "fs": np.asarray(session_parents["fs"].iloc[0], dtype=float),
        f"{prefix}/processed_values": processed_values,
        f"{prefix}/processed_offsets": processed_offsets,
        f"{prefix}/time_values": time_values,
        f"{prefix}/time_offsets": time_offsets,
        f"{prefix}/window_id": session_meta["window_id"].to_numpy(int),
        f"{prefix}/window_30_id": session_meta["window_30_id"].to_numpy(str),
        f"{prefix}/parent_window_60_id": (
            session_meta["parent_window_60_id"].to_numpy(str)
        ),
        f"{prefix}/parent_window_id": (
            session_meta["parent_window_id"].to_numpy(int)
        ),
        f"{prefix}/subwindow": session_meta["subwindow"].to_numpy(str),
        f"{prefix}/start_offset_s": (
            session_meta["start_offset_s"].to_numpy(float)
        ),
        f"{prefix}/end_offset_s": (
            session_meta["end_offset_s"].to_numpy(float)
        ),
        f"{prefix}/label": session_meta["label"].to_numpy(int),
        f"{prefix}/start_time": (
            session_meta["start_time"].to_numpy(float)
        ),
        f"{prefix}/end_time": session_meta["end_time"].to_numpy(float),
        f"{prefix}/n_samples": session_meta["n_samples"].to_numpy(int),
        f"{prefix}/stationarity_score_raw": (
            np.full(len(session_meta), np.nan)
        ),
        f"{prefix}/stationarity_pass_raw": (
            np.zeros(len(session_meta), dtype=bool)
        ),
        f"{prefix}/stationarity_score_processed": (
            np.full(len(session_meta), np.nan)
        ),
        f"{prefix}/stationarity_pass_processed": (
            np.ones(len(session_meta), dtype=bool)
        ),
    }
    np.savez_compressed(archive_path, **payload)
    for child in session_meta.itertuples(index=False):
        index_rows.append(
            {
                "session": f"{session}.csv",
                "npz_file": archive_path.name,
                "row_index": child.row_index,
                "window_id": child.window_id,
                "window_size_s": CHILD_WINDOW_S,
                "label": child.label,
                "start_time": child.start_time,
                "end_time": child.end_time,
                "fs": float(session_parents["fs"].iloc[0]),
                "n_samples": child.n_samples,
                "stationarity_score_raw": np.nan,
                "stationarity_pass_raw": False,
                "stationarity_score_processed": np.nan,
                "stationarity_pass_processed": True,
                "representation": "processed",
                "window_30_id": child.window_30_id,
                "parent_window_60_id": child.parent_window_60_id,
                "parent_window_id": child.parent_window_id,
                "subwindow": child.subwindow,
                "start_offset_s": child.start_offset_s,
                "end_offset_s": child.end_offset_s,
                "source_file": child.source_file,
                "stationarity_origin": (
                    "inherited_from_finalized_parent_60s_cohort"
                ),
                "split_method": (
                    "timestamp_fs_boundary_nonoverlapping_no_trim_pad"
                ),
                "lineage": (
                    "Processed 60s finalized cohort -> 30s h1/h2"
                ),
            }
        )
segments_index = pd.DataFrame(index_rows)
index_path = OUTPUT_DIR / "segments_index.csv"
segments_index.to_csv(index_path, index=False)
print(f"Saved {len(session_archives)} session NPZ archives")
print(f"Saved global index with {len(segments_index)} child rows")
print(f"Output: {OUTPUT_DIR}")
display(segments_index.head(8))


Saved 20 session NPZ archives
Saved global index with 1802 child rows
Output: /home/vutu0809/Desktop/NTSA_Foundation/phase1/segmentated_data/30s_dhdata


,session,npz_file,row_index,window_id,window_size_s,label,start_time,end_time,fs,n_samples,...,window_30_id,parent_window_60_id,parent_window_id,subwindow,start_offset_s,end_offset_s,source_file,stationarity_origin,split_method,lineage
0,sample_1.csv,sample_1.npz,0,1,30,0,2.63,32.61,50.0,1500,...,sample_1_awake_w0001_60s_h1,sample_1_awake_w0001_60s,1,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
1,sample_1.csv,sample_1.npz,1,2,30,0,32.63,62.61,50.0,1500,...,sample_1_awake_w0001_60s_h2,sample_1_awake_w0001_60s,1,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
2,sample_1.csv,sample_1.npz,2,3,30,0,62.63,92.61,50.0,1500,...,sample_1_awake_w0002_60s_h1,sample_1_awake_w0002_60s,2,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
3,sample_1.csv,sample_1.npz,3,4,30,0,92.63,122.61,50.0,1500,...,sample_1_awake_w0002_60s_h2,sample_1_awake_w0002_60s,2,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
4,sample_1.csv,sample_1.npz,4,5,30,0,122.63,152.61,50.0,1500,...,sample_1_awake_w0003_60s_h1,sample_1_awake_w0003_60s,3,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
5,sample_1.csv,sample_1.npz,5,6,30,0,152.63,182.61,50.0,1500,...,sample_1_awake_w0003_60s_h2,sample_1_awake_w0003_60s,3,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
6,sample_1.csv,sample_1.npz,6,7,30,0,182.63,212.61,50.0,1500,...,sample_1_awake_w0004_60s_h1,sample_1_awake_w0004_60s,4,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
7,sample_1.csv,sample_1.npz,7,8,30,0,212.63,242.61,50.0,1500,...,sample_1_awake_w0004_60s_h2,sample_1_awake_w0004_60s,4,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2


In [3]:
# Cell 3 — Integrity QC, exact reconstruction, and get_data reuse test.
qc_rows: list[dict[str, object]] = []


def record_qc(check: str, passed: bool, details: str) -> None:
    """Record one integrity check."""
    qc_rows.append(
        {"check": check, "passed": bool(passed), "details": details}
    )


npz_paths = sorted(OUTPUT_DIR.glob("sample_*.npz"))
all_files = [path for path in OUTPUT_DIR.iterdir() if path.is_file()]
record_qc(
    "canonical_file_layout",
    len(npz_paths) == EXPECTED_SESSIONS
    and len(all_files) == EXPECTED_SESSIONS + 1
    and (OUTPUT_DIR / "segments_index.csv").is_file(),
    f"session_npz={len(npz_paths)}; total_files={len(all_files)}",
)
record_qc(
    "expected_child_count",
    len(segments_index) == 2 * len(parent_index),
    f"observed={len(segments_index)}; expected={2 * len(parent_index)}",
)
parent_counts = segments_index.groupby("parent_window_60_id").size()
parent_halves = segments_index.groupby("parent_window_60_id")[
    "subwindow"
].agg(lambda values: set(values))
record_qc(
    "parent_h1_h2_mapping",
    len(parent_counts) == len(parent_index)
    and parent_counts.eq(2).all()
    and parent_halves.map(lambda halves: halves == {"h1", "h2"}).all(),
    f"parents={len(parent_counts)}; count_failures={parent_counts.ne(2).sum()}",
)
record_qc(
    "processed_only_lineage",
    segments_index["representation"].eq("processed").all()
    and segments_index["stationarity_pass_processed"].all()
    and not segments_index["stationarity_pass_raw"].any(),
    "no raw signal; processed eligibility inherited from parent cohort",
)
child_state_counts = (
    segments_index["label"].map({0: "Awake", 1: "Drowsy"})
    .value_counts()
    .reindex(STATES)
)
record_qc(
    "state_distribution_doubled",
    child_state_counts.eq(2 * input_state_counts).all(),
    f"observed={child_state_counts.to_dict()}",
)
sample_deviation = (
    segments_index["n_samples"]
    - np.rint(30.0 * segments_index["fs"]).astype(int)
).abs()
duration_deviation = (segments_index["n_samples"] / segments_index["fs"] - 30.0).abs()
record_qc(
    "sample_count_and_duration",
    sample_deviation.le(1).all()
    and duration_deviation.le(DURATION_TOLERANCE_S).all(),
    f"max_sample_error={sample_deviation.max()}; "
    f"max_duration_error_s={duration_deviation.max():.6f}",
)
loaded_signals: dict[str, np.ndarray] = {}
loaded_times: dict[str, np.ndarray] = {}
loader_failures: list[str] = []
raw_key_failures: list[str] = []
for npz_path in npz_paths:
    session = npz_path.stem
    try:
        awake, drowsy = get_data(
            session,
            data_dir=OUTPUT_DIR,
            window_sizes=30,
            stationarity="processed",
        )
    except Exception as exc:
        loader_failures.append(f"{session}: {exc}")
        continue
    session_rows = segments_index.loc[
        segments_index["npz_file"].eq(npz_path.name)
    ]
    uid_lookup = session_rows.set_index("window_id")["window_30_id"]
    for state_batch in (awake[30], drowsy[30]):
        if "raw" in state_batch:
            raw_key_failures.append(session)
        for index, window_id in enumerate(state_batch["window_id"]):
            uid = uid_lookup.loc[int(window_id)]
            signal = np.asarray(state_batch["processed"][index], dtype=float)
            time = np.asarray(state_batch["time"][index], dtype=float)
            if (
                signal.ndim != 1
                or signal.size == 0
                or signal.shape != time.shape
                or not np.isfinite(signal).all()
                or not np.isfinite(time).all()
            ):
                loader_failures.append(uid)
            loaded_signals[uid] = signal
            loaded_times[uid] = time
record_qc(
    "get_data_reuse",
    not loader_failures and len(loaded_signals) == len(segments_index),
    f"loaded={len(loaded_signals)}; failures={len(loader_failures)}",
)
record_qc(
    "no_fabricated_raw_signal",
    not raw_key_failures,
    f"failures={len(raw_key_failures)}",
)
reconstruction_failures: list[str] = []
overlap_failures: list[str] = []
for parent_uid, group in segments_index.groupby("parent_window_60_id"):
    group = group.set_index("subwindow").loc[["h1", "h2"]]
    child_uids = group["window_30_id"].tolist()
    if not all(uid in loaded_signals for uid in child_uids):
        reconstruction_failures.append(parent_uid)
        continue
    signals = [loaded_signals[uid] for uid in child_uids]
    times = [loaded_times[uid] for uid in child_uids]
    if not (
        np.array_equal(np.concatenate(signals), parent_signals[parent_uid])
        and np.array_equal(np.concatenate(times), parent_times[parent_uid])
    ):
        reconstruction_failures.append(parent_uid)
    if not times[0][-1] < times[1][0]:
        overlap_failures.append(parent_uid)
record_qc(
    "exact_parent_reconstruction",
    not reconstruction_failures,
    f"failures={len(reconstruction_failures)}",
)
record_qc(
    "sample_level_no_overlap",
    not overlap_failures,
    f"failures={len(overlap_failures)}",
)
parent_lookup = parent_index.set_index("parent_window_60_id")
inherited = segments_index.join(
    parent_lookup[["session", "label", "fs"]],
    on="parent_window_60_id",
    rsuffix="_parent",
)
metadata_ok = (
    inherited["session"].str.removesuffix(".csv").eq(
        inherited["session_parent"]
    ).all()
    and inherited["label"].eq(inherited["label_parent"]).all()
    and np.allclose(inherited["fs"], inherited["fs_parent"])
)
record_qc(
    "metadata_inheritance",
    metadata_ok,
    "session/label/fs match each parent",
)
integrity_report = pd.DataFrame(qc_rows)
require(
    integrity_report["passed"].all(),
    "30-s session-level dataset failed integrity QC.",
)
print(f"60-s parent windows : {len(parent_index)}")
print(f"30-s child windows  : {len(segments_index)}")
print(f"Session NPZ files   : {len(npz_paths)}")
print(f"Global index rows   : {len(segments_index)}")
print(f"Awake               : {int(child_state_counts['Awake'])}")
print(f"Drowsy              : {int(child_state_counts['Drowsy'])}")
print("get_data reuse      : PASS")
print("Reconstruction     : PASS")
print("Dataset status     : READY FOR NTSA")
display(integrity_report)
display(segments_index.head(8))


60-s parent windows : 901
30-s child windows  : 1802
Session NPZ files   : 20
Global index rows   : 1802
Awake               : 1192
Drowsy              : 610
get_data reuse      : PASS
Reconstruction     : PASS
Dataset status     : READY FOR NTSA


,check,passed,details
0,canonical_file_layout,True,session_npz=20; total_files=21
1,expected_child_count,True,observed=1802; expected=1802
2,parent_h1_h2_mapping,True,parents=901; count_failures=0
3,processed_only_lineage,True,no raw signal; processed eligibility inherited...
4,state_distribution_doubled,True,"observed={'Awake': 1192, 'Drowsy': 610}"
5,sample_count_and_duration,True,max_sample_error=1; max_duration_error_s=0.042996
6,get_data_reuse,True,loaded=1802; failures=0
7,no_fabricated_raw_signal,True,failures=0
8,exact_parent_reconstruction,True,failures=0
9,sample_level_no_overlap,True,failures=0


,session,npz_file,row_index,window_id,window_size_s,label,start_time,end_time,fs,n_samples,...,window_30_id,parent_window_60_id,parent_window_id,subwindow,start_offset_s,end_offset_s,source_file,stationarity_origin,split_method,lineage
0,sample_1.csv,sample_1.npz,0,1,30,0,2.63,32.61,50.0,1500,...,sample_1_awake_w0001_60s_h1,sample_1_awake_w0001_60s,1,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
1,sample_1.csv,sample_1.npz,1,2,30,0,32.63,62.61,50.0,1500,...,sample_1_awake_w0001_60s_h2,sample_1_awake_w0001_60s,1,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
2,sample_1.csv,sample_1.npz,2,3,30,0,62.63,92.61,50.0,1500,...,sample_1_awake_w0002_60s_h1,sample_1_awake_w0002_60s,2,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
3,sample_1.csv,sample_1.npz,3,4,30,0,92.63,122.61,50.0,1500,...,sample_1_awake_w0002_60s_h2,sample_1_awake_w0002_60s,2,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
4,sample_1.csv,sample_1.npz,4,5,30,0,122.63,152.61,50.0,1500,...,sample_1_awake_w0003_60s_h1,sample_1_awake_w0003_60s,3,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
5,sample_1.csv,sample_1.npz,5,6,30,0,152.63,182.61,50.0,1500,...,sample_1_awake_w0003_60s_h2,sample_1_awake_w0003_60s,3,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
6,sample_1.csv,sample_1.npz,6,7,30,0,182.63,212.61,50.0,1500,...,sample_1_awake_w0004_60s_h1,sample_1_awake_w0004_60s,4,h1,0.0,30.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
7,sample_1.csv,sample_1.npz,7,8,30,0,212.63,242.61,50.0,1500,...,sample_1_awake_w0004_60s_h2,sample_1_awake_w0004_60s,4,h2,30.0,60.0,sample_1.npz,inherited_from_finalized_parent_60s_cohort,timestamp_fs_boundary_nonoverlapping_no_trim_pad,Processed 60s finalized cohort -> 30s h1/h2
